# Tratamento de Qualidade
Neste notebook, aplicaremos as regras de limpeza, tratamento de ausências e correção de símbolos da nossa base de dados.

# Importação de Bibliotecas

In [25]:
import pandas as pd
import numpy as np

# Carregamento dos Dados
Carregamento das bases brutas para aplicação dos tratamentos.

In [26]:
arquivos_pam = [
    "../data/dados_originais/t5457_pam_area_colhida_2003_2024_ce_br.csv",
    "../data/dados_originais/t5457_pam_area_plantada_2003_2024_ce_br.csv",
    "../data/dados_originais/t5457_pam_quantidade_produzida_2003_2024_ce_br.csv",
    "../data/dados_originais/t5457_pam_rendimento_medio_2003_2024_ce_br.csv",
    "../data/dados_originais/t5457_pam_valor_producao_2003_2024_ce_br.csv"
]

dfs_pam = []
for arquivo in arquivos_pam:
    df = pd.read_csv(arquivo, sep=";", dtype={"territorio_codigo": "string", "variavel_codigo": "string", "produto_codigo": "string"})
    dfs_pam.append(df)
df_pam = pd.concat(dfs_pam, ignore_index=True)

df_ppm = pd.read_csv("../data/dados_originais/t3939_ppm_efetivo_rebanhos_2003_2024_ce_br.csv", sep=";", dtype={"territorio_codigo": "string", "variavel_codigo": "string", "tipo_rebanho_codigo": "string"})
df_pib = pd.read_csv("../data/dados_originais/t5938_pib_agropecuaria_2003_2023_ce_br.csv", sep=";", dtype={"territorio_codigo": "string", "variavel_codigo": "string"})

# Definição das Chaves Únicas
Chaves identificadoras estabelecidas para cada conjunto de dados.

In [27]:
chaves_pam = ['nivel_territorial_codigo', 'territorio_codigo', 'ano_codigo', 'variavel_codigo', 'produto_codigo']
chaves_ppm = ['nivel_territorial_codigo', 'territorio_codigo', 'ano_codigo', 'variavel_codigo', 'tipo_rebanho_codigo']
chaves_pib = ['nivel_territorial_codigo', 'territorio_codigo', 'ano_codigo', 'variavel_codigo']

# Definir regra para duplicatas & Tratar duplicatas

Com base nas diretrizes do projeto, as regras para duplicatas foram definidas avaliando três cenários principais:

1. **Cenário A: Falha Técnica ou de Ingestão (Remoção Obrigatória)** - *Causa:* Duplo envio, erro em chamadas, etc. *Ação:* Remover as duplicatas preservando o registro mais antigo (`keep='first'`) ou mais recente (`keep='last'`).
2. **Cenário B: Eventos Repetidos Legítimos (Agregação)** - *Ação:* Agrupamento (groupby) e contagem.
3. **Cenário C: Duplicação Indesejada em Junções (Merges/Joins)** - *Ação:* Validar a unicidade no cruzamento de tabelas.

**Nossa Regra Aplicada:**
Dado que as bases do IBGE são dados agregados e estruturados, qualquer duplicata encontrada nas chaves primárias recai sob o **Cenário A (Falha de Ingestão)**. Portanto, aplicaremos o método de remoção obrigatória com `keep='last'`, assumindo que a ingestão mais recente de um dado sobrepõe um erro anterior.

> **Nota:** Durante as análises produzidas no notebook de validação (`validacao_qualidade.ipynb`), **não foram encontradas duplicatas nos dados atuais**. O tratamento a seguir atua de forma preventiva caso a base receba novos dados ou atualizações futuramente.

In [28]:
tamanho_antes_pam = len(df_pam)
tamanho_antes_ppm = len(df_ppm)
tamanho_antes_pib = len(df_pib)

df_pam = df_pam.drop_duplicates(subset=chaves_pam, keep='last')
df_ppm = df_ppm.drop_duplicates(subset=chaves_ppm, keep='last')
df_pib = df_pib.drop_duplicates(subset=chaves_pib, keep='last')

print(f"Duplicatas removidas PAM: {tamanho_antes_pam - len(df_pam)}")
print(f"Duplicatas removidas PPM: {tamanho_antes_ppm - len(df_ppm)}")
print(f"Duplicatas removidas PIB: {tamanho_antes_pib - len(df_pib)}")

Duplicatas removidas PAM: 0
Duplicatas removidas PPM: 0
Duplicatas removidas PIB: 0


# Definir regra para ausências & Tratar símbolos especiais

Com base nas diretrizes de tratamento estabelecidas, aplicaremos as seguintes transformações na coluna `valor`:

- **`-` (Hífen = Zero Absoluto)**: Converter para `0.0`. Representa que o evento foi observado e não ocorreu de fato. Tratar como NaN alteraria incorretamente a contagem e a média dos dados.
- **`0` (Zero)**: Manter como `0.0`. Número válido.
- **`X` (Dado Inibido)**: Converter para `NaN` e criar flag binária `valor_is_inibido` indicando `1` para dado omitido por sigilo estatístico.
- **`..` (Não se aplica)**: Converter para `NaN` (pois nossa coluna é numérica). Evento não tem cabimento lógico.
- **`...` (Dado não disponível)**: Converter para `NaN`. Ausência documentada clássica.

In [29]:
def tratar_simbolos_ausencias(df):
    # Garantir que a coluna valor seja string para o tratamento inicial
    df['valor'] = df['valor'].astype(str).str.strip()
    
    # 1. Criar a flag binária para 'X' (Dado Inibido)
    df['valor_is_inibido'] = np.where(df['valor'] == 'X', 1, 0)
    
    # 2. Aplicar as regras de substituição
    substituicoes = {
        '-': '0.0',
        '0': '0.0',
        'X': np.nan,
        '..': np.nan,
        '...': np.nan
    }
    df['valor'] = df['valor'].replace(substituicoes)
    
    # 3. Converter para numérico
    # errors='coerce' transformará qualquer outro texto não numérico que tenha passado em NaN
    df['valor'] = pd.to_numeric(df['valor'], errors='coerce')
    
    return df

df_pam = tratar_simbolos_ausencias(df_pam)
df_ppm = tratar_simbolos_ausencias(df_ppm)
df_pib = tratar_simbolos_ausencias(df_pib)

print("Tratamento concluído com sucesso.")
print(f"PAM - Valores nulos (ausentes + especiais) após tratamento: {df_pam['valor'].isna().sum()} | Flag Inibidos (X): {df_pam['valor_is_inibido'].sum()}")
print(f"PPM - Valores nulos (ausentes + especiais) após tratamento: {df_ppm['valor'].isna().sum()} | Flag Inibidos (X): {df_ppm['valor_is_inibido'].sum()}")
print(f"PIB - Valores nulos (ausentes + especiais) após tratamento: {df_pib['valor'].isna().sum()} | Flag Inibidos (X): {df_pib['valor_is_inibido'].sum()}")

Tratamento concluído com sucesso.
PAM - Valores nulos (ausentes + especiais) após tratamento: 0 | Flag Inibidos (X): 0
PPM - Valores nulos (ausentes + especiais) após tratamento: 0 | Flag Inibidos (X): 0
PIB - Valores nulos (ausentes + especiais) após tratamento: 1116 | Flag Inibidos (X): 0


# Validação Pós-Tratamento
Agora que a coluna `valor` é estritamente numérica e os símbolos (`X`, `...`, etc) foram tratados, podemos validar a integridade matemática das agregações hierárquicas da base do IBGE.

A validação consistirá em:
1. Isolar os registros de Municípios (código `N6`).
2. Agrupar e somar os valores numéricos dos municípios por UF (através dos 2 primeiros dígitos do código municipal) e Ano.
3. Isolar os registros oficiais de UF (código `N3`).
4. Comparar a soma municipal com o total estadual oficial para encontrar furos/inconsistências.

In [30]:
def validar_agregacao_uf_municipio(df, nome_tabela):
    print(f"--- Validação de Agregação: {nome_tabela} ---")
    
    # Garantir que as chaves comparativas são strings
    df['nivel_territorial_codigo'] = df['nivel_territorial_codigo'].astype(str)
    df['territorio_codigo'] = df['territorio_codigo'].astype(str)
    
    # 1. Isolar Estados (UF = Nível N3 no IBGE SIDRA)
    df_uf = df[df['nivel_territorial_codigo'] == 'N3'].copy()
    
    # 2. Isolar Municípios (Nível N6 no IBGE SIDRA)
    df_mun = df[df['nivel_territorial_codigo'] == 'N6'].copy()
    
    if df_mun.empty or df_uf.empty:
        print("Não há dados suficientes de Estado (UF) e Municípios simultaneamente para rodar a validação nesta base.\n")
        return
        
    # O código da UF são os 2 primeiros caracteres do código do Município
    df_mun['uf_codigo_extraido'] = df_mun['territorio_codigo'].str[:2]
    
    # Determinar colunas dinâmicas para o GroupBy
    colunas_agrupamento = ['ano_codigo', 'variavel_codigo', 'uf_codigo_extraido']
    if 'produto_codigo' in df_mun.columns:
        colunas_agrupamento.append('produto_codigo')
    if 'tipo_rebanho_codigo' in df_mun.columns:
        colunas_agrupamento.append('tipo_rebanho_codigo')
        
    # Agrupar e somar os valores (ignorando NaNs via .sum() automático do pandas)
    mun_agrupado = df_mun.groupby(colunas_agrupamento, as_index=False)['valor'].sum()
    mun_agrupado.rename(columns={'valor': 'soma_municipios', 'uf_codigo_extraido': 'territorio_codigo'}, inplace=True)
    
    # 3. Cruzar com os dados oficiais da UF
    # Ajusta as chaves de cruzamento renomeando uf_codigo_extraido de volta para territorio_codigo
    chaves_merge = [col if col != 'uf_codigo_extraido' else 'territorio_codigo' for col in colunas_agrupamento]
    
    comparacao = pd.merge(df_uf, mun_agrupado, on=chaves_merge, how='inner')
    
    # 4. Calcular diferenças absolutas
    comparacao['diferenca_absoluta'] = (comparacao['valor'] - comparacao['soma_municipios']).abs()
    inconsistencias = comparacao[comparacao['diferenca_absoluta'] > 0.01] # margem de segurança para floats
    
    print(f"Total de agregações validadas (Cenários Únicos): {len(comparacao)}")
    print(f"Inconsistências encontradas (Soma dos Municípios != Total Oficial da UF): {len(inconsistencias)}")
    
    if not inconsistencias.empty:
        print("\nExemplo das 5 maiores inconsistências (Possivelmente causadas por dados 'X' inibidos ou ausentes):")
        display = inconsistencias.sort_values(by='diferenca_absoluta', ascending=False).head(5)
        print(display[chaves_merge + ['valor', 'soma_municipios', 'diferenca_absoluta']].to_string(index=False))
    print("\n" + "-"*60 + "\n")

validar_agregacao_uf_municipio(df_pam, 'PAM')
validar_agregacao_uf_municipio(df_ppm, 'PPM')
validar_agregacao_uf_municipio(df_pib, 'PIB')

--- Validação de Agregação: PAM ---
Total de agregações validadas (Cenários Únicos): 770
Inconsistências encontradas (Soma dos Municípios != Total Oficial da UF): 298

Exemplo das 5 maiores inconsistências (Possivelmente causadas por dados 'X' inibidos ou ausentes):
 ano_codigo variavel_codigo territorio_codigo produto_codigo   valor  soma_municipios  diferenca_absoluta
       2008             112                23          40106 53863.0        5936962.0           5883099.0
       2003             112                23          40106 52317.0        5927406.0           5875089.0
       2004             112                23          40106 51872.0        5899436.0           5847564.0
       2006             112                23          40106 55630.0        5888671.0           5833041.0
       2005             112                23          40106 50918.0        5847873.0           5796955.0

------------------------------------------------------------

--- Validação de Agregação: PPM --

# Documentação das Decisões de Tratamento e Validação

Com base nos testes e diagnósticos realizados acima, firmamos as seguintes decisões para a garantia da qualidade e integridade analítica das bases PAM, PPM e PIB:

### 1. Tratamento de Duplicatas
- **Decisão**: Adoção da remoção obrigatória preventivamente (`keep='last'`).
- **Justificativa**: Por se tratar de bases agregadas governamentais (IBGE), a duplicidade nas chaves primárias (Ano, Município, Produto) caracteriza **Falha de Ingestão ou Atualização** (Cenário A). Assumimos que o registro mais recente (último ingerido) corrige o anterior.

### 2. Tratamento de Símbolos Especiais e Ausências
- **`-` e `0`**: Convertidos para numérico `0.0`. Representam eventos observados cuja contagem é matematicamente nula (ex: não houve plantio naquele ano).
- **`X` (Sigilo Estatístico)**: Convertido para `NaN` (nulo) **com a criação da flag indicadora `valor_is_inibido`**. Ocultar o dado por sigilo é uma informação estrutural relevante, e a exclusão do registro sem documentar a ocorrência distorceria modelos analíticos.
- **`..` e `...` (Não se aplica / Não disponível)**: Convertidos para `NaN` clássico para não distorcer o cálculo de médias/medianas das séries temporais.

### 3. Validação Hierárquica (Municípios vs. UF)
- **Conclusão**: A soma dos valores dos municípios (`N6`) **não iguala** perfeitamente ao total oficial da UF (`N3`) em determinados recortes.
- **Justificativa**: A principal causa para essas inconsistências é justamente a presença de dados inibidos (`X`) no nível municipal. O valor existe e é contabilizado no fechamento macro (UF e Brasil), mas é censurado no micro (Município) por LGPD ou sigilo de mercado. 
- **Diretriz**: Para análises macroeconômicas ou estaduais, **deve-se utilizar exclusivamente a linha oficial da UF**, e não a agregação manual dos municípios tratados, que possui "buracos".

# Comparativo Antes e Depois (Resumo do Pipeline)

Com o fim do pipeline de tratamento de qualidade, fazemos o balanço final do estado da base para prosseguirmos à modelagem:
- **Estrutura (Schema)**: A coluna `valor` iniciou como tipo `object` (string) devido aos símbolos estatísticos, e agora está propriamente tratada como numérica (`float64`).
- **Engenharia de Atributos**: Adicionamos 1 nova coluna (`valor_is_inibido`) que sinaliza (1 ou 0) a presença de dados outrora omitidos por sigilo estatístico (`X`), útil para modelos de regressão ou detecção de anomalias.
- **Ausências (Nulls)**: Símbolos que representavam mascaramentos textuais (como `...`) foram convertidos formalmente para `NaN` do Pandas. Isso garante que a exclusão desse dado em agregações seja matemática e automática, sem causar falhas de conversão de tipagem.

In [31]:
print("================ ANÁLISE FINAL: PAM ================")
print(df_pam.info())
print("\n--- Resumo Estatístico ---")
print(df_pam[['valor', 'valor_is_inibido']].describe())

print("\n\n================ ANÁLISE FINAL: PPM ================")
print(df_ppm.info())
print("\n--- Resumo Estatístico ---")
print(df_ppm[['valor', 'valor_is_inibido']].describe())

print("\n\n================ ANÁLISE FINAL: PIB ================")
print(df_pib.info())
print("\n--- Resumo Estatístico ---")
print(df_pib[['valor', 'valor_is_inibido']].describe())

================ ANÁLISE FINAL: PAM ================
<class 'pandas.DataFrame'>
RangeIndex: 143220 entries, 0 to 143219
Data columns (total 13 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   nivel_territorial_codigo  143220 non-null  str    
 1   nivel_territorial_nome    143220 non-null  str    
 2   territorio_codigo         143220 non-null  str    
 3   territorio_nome           143220 non-null  str    
 4   ano_codigo                143220 non-null  int64  
 5   ano_nome                  143220 non-null  int64  
 6   variavel_codigo           143220 non-null  string 
 7   variavel_nome             143220 non-null  str    
 8   unidade                   143220 non-null  str    
 9   produto_codigo            143220 non-null  string 
 10  produto_nome              143220 non-null  str    
 11  valor                     143220 non-null  float64
 12  valor_is_inibido          143220 non-null  int64  
dtypes: